In [1]:
import os
print("Installing required packages... (this may take a few minutes)")

# Only execute installs if inside a Jupyter-style environment (Kaggle)
try:
    get_ipython  # type: ignore
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False

if IN_NOTEBOOK:
    # system packages for OCR and PDF image extraction
    !apt-get update -qq && apt-get install -y -qq tesseract-ocr poppler-utils
    # pin protobuf to avoid MessageFactory/GetPrototype errors
    !pip install -q --upgrade pip
    !pip install -q protobuf==3.20.3
    # python libs
    !pip install -q pdfplumber pytesseract pillow sentence-transformers transformers faiss-cpu gradio umap-learn matplotlib scikit-learn tabulate rouge-score joblib accelerate

print("Installs completed. If the protobuf install ran, restart the kernel before re-running the cell.")


Installing required packages... (this may take a few minutes)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 128639 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.12) over (22.02.0-2ubuntu0.8) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.12) over (22.02.0-2ubuntu0.8) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.12) over (22.02.0-2ubuntu0.8) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up libpop

In [2]:
# 2) Imports & config
import io
import json
import uuid
import time
import math
import faiss
import joblib
import numpy as np
import pdfplumber
import pytesseract
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel, pipeline, AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import umap
import matplotlib.pyplot as plt
import pandas as pd
import gradio as gr

# Paths (Kaggle dataset path)
PDF_PATHS = [
    '/kaggle/input/a3-data/1. Annual Report 2023-24.pdf',
    '/kaggle/input/a3-data/2. financials.pdf',
    '/kaggle/input/a3-data/3. FYP-Handbook-2023.pdf'
]

WORK_DIR = '/kaggle/working/a3_multimodal'
IMAGES_DIR = os.path.join(WORK_DIR, 'images')
INDEX_DIR = os.path.join(WORK_DIR, 'index')
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

TEXT_EMBED_MODEL = 'all-MiniLM-L6-v2'  # small and fast
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'  # optional, may fail to load - code handles fallback
EMBED_NPY = os.path.join(INDEX_DIR, 'embeddings.npy')
METADATA_PATH = os.path.join(INDEX_DIR, 'metadata.json')
FAISS_INDEX_PATH = os.path.join(INDEX_DIR, 'faiss.index')
PCA_TEXT_PATH = os.path.join(INDEX_DIR, 'pca_text.joblib')
PCA_IMG_PATH = os.path.join(INDEX_DIR, 'pca_img.joblib')

# chunk params
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

# LLM config (OpenAI key optional; HF fallback used otherwise)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
HF_LLM = 'gpt2'  # small fallback for demos

# 3) PDF parsing + OCR + chunking helpers
def parse_pdf(pdf_path):
    elements = []
    src = os.path.basename(pdf_path)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ''
                if text.strip():
                    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
                    for p in paras:
                        elements.append({'id': str(uuid.uuid4()), 'source': src, 'page': i, 'type': 'paragraph', 'content': p})
                # tables
                try:
                    tables = page.extract_tables() or []
                except Exception:
                    tables = []
                for t in tables:
                    if not t:
                        continue
                    df = pd.DataFrame(t[1:], columns=t[0]) if len(t) > 1 else pd.DataFrame(t)
                    tabstr = df.to_csv(index=False)
                    elements.append({'id': str(uuid.uuid4()), 'source': src, 'page': i, 'type': 'table', 'content': tabstr})
                # images via page.images bbox cropping
                try:
                    for img_idx, img in enumerate(page.images or []):
                        bbox = (img['x0'], img['top'], img['x1'], img['bottom'])
                        try:
                            cropped = page.within_bbox(bbox).to_image(resolution=150)
                            buf = io.BytesIO()
                            cropped.original.save(buf, format='PNG')
                            img_bytes = buf.getvalue()
                            img_path = os.path.join(IMAGES_DIR, f"{src}_p{i}_img{img_idx}.png")
                            with open(img_path, 'wb') as f:
                                f.write(img_bytes)
                            elements.append({'id': str(uuid.uuid4()), 'source': src, 'page': i, 'type': 'image', 'content': img_path})
                        except Exception:
                            continue
                except Exception:
                    pass
    except Exception as e:
        print("Failed to open PDF:", pdf_path, e)
    return elements

def ocr_image(path):
    try:
        img = Image.open(path).convert('RGB')
        return pytesseract.image_to_string(img)
    except Exception as e:
        print("OCR error:", e)
        return ""

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    tokens = text.split()
    if len(tokens) <= size:
        return [text]
    chunks = []
    i = 0
    while i < len(tokens):
        chunks.append(' '.join(tokens[i:i+size]))
        i += size - overlap
    return chunks

def create_chunks(elements):
    chunks = []
    for el in elements:
        if el['type'] in ('paragraph', 'table'):
            for c in chunk_text(el['content']):
                chunks.append({'id': str(uuid.uuid4()), 'source': el['source'], 'page': el['page'], 'type': el['type'], 'text': c})
        elif el['type'] == 'image':
            img_path = el['content']
            ocr = ocr_image(img_path)
            chunks.append({'id': str(uuid.uuid4()), 'source': el['source'], 'page': el['page'], 'type': 'image_ocr', 'text': ocr, 'image_path': img_path})
            chunks.append({'id': str(uuid.uuid4()), 'source': el['source'], 'page': el['page'], 'type': 'image', 'text': '', 'image_path': img_path})
    return chunks

# 4) Load embedding models
print("Loading text embedder:", TEXT_EMBED_MODEL)
text_embedder = SentenceTransformer(TEXT_EMBED_MODEL)

print("Loading CLIP model (optional):", CLIP_MODEL_NAME)
try:
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
    clip_available = True
    print("CLIP loaded.")
except Exception as e:
    print("CLIP failed to load:", e)
    clip_model = None
    clip_processor = None
    clip_available = False

import torch
def embed_texts(texts, batch_size=32):
    return np.array(text_embedder.encode(texts, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=False))

def embed_image(path):
    if not clip_available:
        raise RuntimeError("CLIP not available")
    image = Image.open(path).convert('RGB')
    inputs = clip_processor(images=image, return_tensors='pt')
    with torch.no_grad():
        feats = clip_model.get_image_features(**inputs)
    vec = feats[0].cpu().numpy()
    vec = vec / np.linalg.norm(vec)
    return vec

# 5) Build index (with shared PCA projection)
def build_index(shared_target=128):
    # 1) Parse PDFs -> elements -> chunks
    all_elements = []
    for p in PDF_PATHS:
        if os.path.exists(p):
            els = parse_pdf(p)
            all_elements.extend(els)
        else:
            print("Missing PDF:", p)
    print("Parsed elements:", len(all_elements))

    chunks = create_chunks(all_elements)
    print("Chunks:", len(chunks))

    # 2) Text embeddings
    texts = [c.get('text','') or '' for c in chunks]
    print("Embedding text chunks...")
    text_embs = embed_texts(texts)  # shape: (n_chunks, text_dim)

    # 3) Image embeddings (only for image-only chunks)
    image_indices = [i for i,c in enumerate(chunks) if c['type']=='image' and c.get('image_path')]
    img_vecs = None
    if clip_available and len(image_indices) > 0:
        print("Embedding image chunks:", len(image_indices))
        img_list = []
        for i in image_indices:
            try:
                v = embed_image(chunks[i]['image_path'])
                img_list.append(v)
            except Exception as e:
                # fallback to zeros
                img_list.append(np.zeros((text_embs.shape[1],), dtype=np.float32))
        img_vecs = np.vstack(img_list)  # shape: (n_image_chunks, img_dim)

    # 4) Decide safe SHARED_DIM based on available samples
    if img_vecs is not None:
        max_components = min(img_vecs.shape[0], text_embs.shape[1])
        SHARED_DIM = min(shared_target, max_components)
    else:
        # no images; limit by text emb features and samples
        SHARED_DIM = min(shared_target, text_embs.shape[1], text_embs.shape[0])
    if SHARED_DIM <= 0:
        raise ValueError("Computed SHARED_DIM is <= 0; adjust shared_target or check inputs.")
    print(f"Using SHARED_DIM = {SHARED_DIM}")

    # 5) Fit PCA on text embeddings to reduce them to SHARED_DIM
    pca_text = PCA(n_components=SHARED_DIM)
    text_embs_reduced = pca_text.fit_transform(text_embs)  # (n_chunks, SHARED_DIM)
    joblib.dump(pca_text, PCA_TEXT_PATH)
    print("Saved PCA for text:", PCA_TEXT_PATH)

    # 6) If we have image vectors, fit PCA for them too and map into SHARED_DIM
    if img_vecs is not None:
        # PCA constraints: n_components <= min(n_samples, n_features); ensure SHARED_DIM <= min(...)
        max_allowed = min(img_vecs.shape[0], img_vecs.shape[1])
        if SHARED_DIM > max_allowed:
            # fallback reduce SHARED_DIM
            old = SHARED_DIM
            SHARED_DIM = max_allowed
            print(f"Adjusted SHARED_DIM to {SHARED_DIM} due to image sample limits (was {old})")
            # reproject text PCA to new SHARED_DIM (re-fit)
            pca_text = PCA(n_components=SHARED_DIM)
            text_embs_reduced = pca_text.fit_transform(text_embs)
            joblib.dump(pca_text, PCA_TEXT_PATH)
            print("Re-saved PCA for text with adjusted SHARED_DIM:", PCA_TEXT_PATH)

        pca_img = PCA(n_components=SHARED_DIM)
        img_vecs_reduced = pca_img.fit_transform(img_vecs)  # (n_image_chunks, SHARED_DIM)
        joblib.dump(pca_img, PCA_IMG_PATH)
        print("Saved PCA for images:", PCA_IMG_PATH)

        # 7) Replace corresponding rows in text_embs_reduced with image-projection
        for idx_loc, emb_vec in zip(image_indices, img_vecs_reduced):
            text_embs_reduced[idx_loc] = emb_vec

    # 8) Final embeddings to index (float32)
    final_embs = text_embs_reduced.astype('float32')
    np.save(EMBED_NPY, final_embs)
    print("Saved reduced embeddings to:", EMBED_NPY)

    # 9) Save metadata
    metadata = []
    for c in chunks:
        metadata.append({
            'id': c['id'],
            'source': c['source'],
            'page': c['page'],
            'type': c['type'],
            'text': c.get('text',''),
            'image_path': c.get('image_path', None)
        })
    with open(METADATA_PATH, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print("Saved metadata:", METADATA_PATH)

    # 10) Build FAISS index (inner product on normalized vectors approximates cosine)
    emb = final_embs
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print("Saved FAISS index:", FAISS_INDEX_PATH, " dimension:", emb.shape[1], " items:", len(metadata))

    return metadata, index

# Build or load index + metadata + PCA
if not (os.path.exists(FAISS_INDEX_PATH) and os.path.exists(METADATA_PATH) and os.path.exists(EMBED_NPY) and os.path.exists(PCA_TEXT_PATH)):
    print("Index or PCA not found: building index (this may take a few minutes)...")
    metadata, index = build_index(shared_target=128)  # recommended target dim
else:
    print("Loading saved index, embeddings, metadata, PCA")
    metadata = json.load(open(METADATA_PATH, 'r', encoding='utf-8'))
    index = faiss.read_index(FAISS_INDEX_PATH)
    print("Loaded index, dimension:", index.d, "items:", len(metadata))

# Load PCA models
pca_text = joblib.load(PCA_TEXT_PATH)
pca_img = joblib.load(PCA_IMG_PATH) if os.path.exists(PCA_IMG_PATH) else None

# 6) Retrieval helpers (apply PCA to queries before searching)
def search_text(query, top_k=5):
    qv = embed_texts([query]).astype('float32')  # (1, text_dim)
    # reduce via PCA same as training
    qv_reduced = pca_text.transform(qv).astype('float32')
    faiss.normalize_L2(qv_reduced)
    D, I = index.search(qv_reduced, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        m = metadata[idx].copy()
        m['score'] = float(score)
        results.append(m)
    return results

def search_image(path, top_k=5):
    if not clip_available:
        return [{'error': 'CLIP not available on server'}]
    qv = embed_image(path).astype('float32').reshape(1, -1)
    # reduce with pca_img if available
    if pca_img is not None:
        qv_reduced = pca_img.transform(qv).astype('float32')
    else:
        # if no image PCA, try project with text PCA shape by padding/truncation (rare)
        qv_reduced = pca_text.transform(qv).astype('float32')
    faiss.normalize_L2(qv_reduced)
    D, I = index.search(qv_reduced, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        m = metadata[idx].copy()
        m['score'] = float(score)
        results.append(m)
    return results
    
# ===== RETRIEVAL EVALUATION METRICS =====
def precision_at_k(retrieved_ids, relevant_ids, k):
    """Calculate Precision@K"""
    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = len(set(retrieved_at_k) & set(relevant_ids))
    return relevant_retrieved / k if k > 0 else 0.0

def recall_at_k(retrieved_ids, relevant_ids, k):
    """Calculate Recall@K"""
    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = len(set(retrieved_at_k) & set(relevant_ids))
    total_relevant = len(relevant_ids)
    return relevant_retrieved / total_relevant if total_relevant > 0 else 0.0

def average_precision(retrieved_ids, relevant_ids):
    """Calculate Average Precision"""
    if not relevant_ids:
        return 0.0
    
    score = 0.0
    num_correct = 0
    
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            num_correct += 1
            score += num_correct / (i + 1)
    
    return score / len(relevant_ids) if len(relevant_ids) > 0 else 0.0

def mean_average_precision(all_retrieved, all_relevant):
    """Calculate Mean Average Precision (MAP)"""
    ap_scores = []
    for retrieved_ids, relevant_ids in zip(all_retrieved, all_relevant):
        ap_scores.append(average_precision(retrieved_ids, relevant_ids))
    return np.mean(ap_scores) if ap_scores else 0.0

def evaluate_retrieval_quality(test_queries, ground_truth, top_k_values=[1, 3, 5, 10]):
    """
    Comprehensive retrieval evaluation
    
    Args:
        test_queries: List of query strings
        ground_truth: List of lists, each containing relevant document IDs for corresponding query
        top_k_values: Different K values to evaluate
    """
    results = {}
    
    all_retrieved_ids = []
    all_relevant_ids = ground_truth
    
    # Run retrieval for all test queries
    for query in test_queries:
        search_results = search_text(query, top_k=max(top_k_values))
        retrieved_ids = [result['id'] for result in search_results]
        all_retrieved_ids.append(retrieved_ids)
    
    # Calculate metrics for each top_k
    for k in top_k_values:
        precisions = []
        recalls = []
        
        for retrieved_ids, relevant_ids in zip(all_retrieved_ids, all_relevant_ids):
            precisions.append(precision_at_k(retrieved_ids, relevant_ids, k))
            recalls.append(recall_at_k(retrieved_ids, relevant_ids, k))
        
        results[f'P@{k}'] = np.mean(precisions)
        results[f'R@{k}'] = np.mean(recalls)
    
    # Calculate MAP
    results['MAP'] = mean_average_precision(all_retrieved_ids, all_relevant_ids)
    
    return results, all_retrieved_ids

# ===== SAMPLE GROUND TRUTH CREATION =====
def create_sample_ground_truth(metadata, num_test_queries=5):
    """
    Create sample ground truth for demonstration
    In practice, you would manually annotate this
    """
    # Sample queries related to your PDF content
    sample_queries = [
        "financial performance 2023",
        "academic policies",
        "research methodology", 
        "budget allocation",
        "student guidelines"
    ]
    
    ground_truth = []
    
    # For demo: automatically create ground truth by searching for exact matches
    for query in sample_queries:
        results = search_text(query, top_k=10)
        # Take top 3 as "relevant" for demo purposes
        relevant_ids = [r['id'] for r in results[:3]]
        ground_truth.append(relevant_ids)
    
    return sample_queries, ground_truth

# ===== EVALUATION INTERFACE =====
def run_retrieval_evaluation():
    """Run comprehensive retrieval evaluation and display results"""
    print("Creating sample ground truth...")
    test_queries, ground_truth = create_sample_ground_truth(metadata)
    
    print("Running retrieval evaluation...")
    results, all_retrieved = evaluate_retrieval_quality(test_queries, ground_truth)
    
    # Display results
    print("\n" + "="*50)
    print("RETRIEVAL EVALUATION RESULTS")
    print("="*50)
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    
    # Create visualization
    plt.figure(figsize=(10, 6))
    
    # Extract P@K and R@K values
    k_values = [1, 3, 5, 10]
    p_scores = [results[f'P@{k}'] for k in k_values]
    r_scores = [results[f'R@{k}'] for k in k_values]
    
    plt.plot(k_values, p_scores, 'bo-', label='Precision@K', linewidth=2)
    plt.plot(k_values, r_scores, 'ro-', label='Recall@K', linewidth=2)
    plt.xlabel('K')
    plt.ylabel('Score')
    plt.title('Retrieval Performance vs K')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    eval_plot_path = os.path.join(WORK_DIR, "retrieval_evaluation.png")
    plt.savefig(eval_plot_path)
    print(f"\nEvaluation plot saved to: {eval_plot_path}")
    
    return results

# 7) LLM answer generation (OpenAI optional, HF fallback)
# ===== ADVANCED PROMPTING TECHNIQUES =====
def zero_shot_prompt(question, context):
    """Zero-shot prompting"""
    return f"""
Question: {question}

Context: {context}

Answer the question based only on the context provided. Be concise and accurate.
Answer:"""

def few_shot_prompt(question, context):
    """Few-shot prompting with examples"""
    return f"""
Example 1:
Question: What was the revenue growth?
Context: The company achieved 15% revenue growth in 2023, reaching $50 million in total revenue.
Answer: The revenue growth was 15% in 2023.

Example 2:
Question: What are the admission requirements?
Context: Admission requires a bachelor's degree with minimum 3.0 GPA and two recommendation letters.
Answer: Admission requires a bachelor's degree with minimum 3.0 GPA and two recommendation letters.

Now answer this question:
Question: {question}
Context: {context}
Answer:"""

def chain_of_thought_prompt(question, context):
    """Chain-of-Thought prompting"""
    return f"""
Question: {question}

Context: {context}

Let's think step by step:

1. First, identify the key elements of the question: what specific information is being asked for?
2. Next, scan the context for relevant information that addresses each part of the question.
3. Then, synthesize the information from different parts of the context if needed.
4. Finally, formulate a comprehensive answer that directly addresses the question.

After this reasoning process, here's the answer:"""

def zero_shot_cot_prompt(question, context):
    """Zero-shot Chain-of-Thought"""
    return f"""
Question: {question}

Context: {context}

Let's think step by step to answer this question based on the context provided.

Step-by-step reasoning:"""

    
def generate_answer_advanced(question, results, prompting_technique="zero_shot"):
    """
    Enhanced LLM answer generation with multiple prompting techniques
    
    Args:
        prompting_technique: one of ["zero_shot", "few_shot", "cot", "zero_shot_cot"]
    """
    context = make_context_from_results(results, top_k=7)
    
    # Select prompting technique
    if prompting_technique == "zero_shot":
        prompt = zero_shot_prompt(question, context)
    elif prompting_technique == "few_shot":
        prompt = few_shot_prompt(question, context)
    elif prompting_technique == "cot":
        prompt = chain_of_thought_prompt(question, context)
    elif prompting_technique == "zero_shot_cot":
        prompt = zero_shot_cot_prompt(question, context)
    else:
        prompt = zero_shot_prompt(question, context)
    
    # Track which technique was used
    technique_used = prompting_technique
    
    # OpenAI
    if OPENAI_API_KEY:
        import openai
        openai.api_key = OPENAI_API_KEY
        try:
            resp = openai.Completion.create(
                engine='text-davinci-003',
                prompt=prompt,
                max_tokens=400,
                temperature=0.3
            )
            return resp.choices[0].text.strip(), technique_used
        except Exception as e:
            print("OpenAI error:", e)

    # HuggingFace fallback
    try:
        from transformers import pipeline
        gen = pipeline('text-generation', model=HF_LLM, tokenizer=HF_LLM)
        out = gen(prompt, max_new_tokens=300, do_sample=False)
        return out[0]['generated_text'][len(prompt):].strip(), technique_used
    except Exception as e:
        print("HF LLM error:", e)
        return "LLM generation unavailable", technique_used

# ===== PROMPTING TECHNIQUE COMPARISON =====
def compare_prompting_techniques(question, results, techniques=None):
    """
    Compare different prompting techniques side-by-side
    """
    if techniques is None:
        techniques = ["zero_shot", "few_shot", "cot", "zero_shot_cot"]
    
    comparisons = []
    
    for technique in techniques:
        start_time = time.time()
        answer, technique_used = generate_answer_advanced(question, results, technique)
        latency = time.time() - start_time
        
        comparisons.append({
            'technique': technique,
            'answer': answer,
            'latency': latency,
            'technique_used': technique_used
        })
    
    return comparisons

def analyze_prompting_impact(test_questions, techniques=None):
    """
    Analyze the impact of different prompting techniques
    """
    if techniques is None:
        techniques = ["zero_shot", "few_shot", "cot", "zero_shot_cot"]
    
    analysis_results = {}
    
    for technique in techniques:
        technique_scores = []
        technique_latencies = []
        
        for question in test_questions:
            results = search_text(question, top_k=5)
            start_time = time.time()
            answer, _ = generate_answer_advanced(question, results, technique)
            latency = time.time() - start_time
            
            # Simple quality metrics (in practice, use human evaluation)
            answer_length = len(answer.split())
            has_keywords = any(keyword in answer.lower() for keyword in question.lower().split())
            
            score = answer_length * 0.1 + (100 if has_keywords else 0)
            technique_scores.append(score)
            technique_latencies.append(latency)
        
        analysis_results[technique] = {
            'avg_score': np.mean(technique_scores),
            'avg_latency': np.mean(technique_latencies),
            'std_score': np.std(technique_scores)
        }
    
    return analysis_results

# 8) Gradio UI (inline)
def make_context_from_results(results, top_k=7):
    """
    Build a cleaned, deduplicated context string from retrieved chunks.
    Removes repeated lines, line breaks, and page breaks.
    """
    context_texts = []
    seen = set()
    
    for r in results[:top_k]:
        if isinstance(r, dict):
            txt = str(r.get('text','') or '')
        else:
            txt = str(r or '')
        
        # Clean text
        txt = txt.replace('\n',' ').replace('\r',' ').replace('\f',' ').strip()
        txt = ' '.join(txt.split())  # remove extra spaces
        
        if txt and txt not in seen:
            context_texts.append(txt)
            seen.add(txt)
    
    return "\n\n".join(context_texts)


# ===== ENHANCED QUERY HANDLER =====
def enhanced_query_handler(query, image, top_k=5, use_llm=True, prompting_technique="zero_shot", compare_techniques=False):
    """
    Enhanced query handler with advanced features
    """
    import time, os
    start = time.time()
    
    # Retrieve relevant chunks
    if image is not None:
        img_path = os.path.join(WORK_DIR, f"user_query_{int(time.time())}.png")
        image.save(img_path)
        results = search_image(img_path, top_k=top_k)
    else:
        results = search_text(query or "", top_k=top_k)
    
    # Generate answer with advanced techniques
    technique_comparisons = None
    if use_llm:
        if compare_techniques:
            technique_comparisons = compare_prompting_techniques(query or "Describe the image", results)
            # Use the first technique as primary answer
            answer = technique_comparisons[0]['answer']
            technique_used = technique_comparisons[0]['technique_used']
        else:
            answer, technique_used = generate_answer_advanced(query or "Describe the image", results, prompting_technique)
    else:
        answer = ""
        technique_used = ""
    
    latency = time.time() - start

    # Format ranked HTML results
    ranked_html = format_results_html(results)
    
    # Add technique comparison results if available
    comparison_html = ""
    if technique_comparisons:
        comparison_html = "<h3>Prompting Technique Comparison</h3>"
        for comp in technique_comparisons:
            comparison_html += f"""
            <div style='border:1px solid #ccc; padding:10px; margin:5px; border-radius:5px; background-color:#f9f9f9'>
                <b>Technique: {comp['technique'].upper()}</b> (Latency: {comp['latency']:.2f}s)<br>
                <small>{comp['answer']}</small>
            </div>
            """
    
    return ranked_html, answer, f"Latency: {latency:.3f}s | Technique: {technique_used}", comparison_html

def format_results_html(results):
    """Format search results as HTML"""
    ranked_html = ""
    for i, r in enumerate(results, start=1):
        if isinstance(r, dict):
            snippet = (r.get('text','') or '')[:400].replace('\n',' ')
            source = r.get('source','?')
            page = r.get('page','?')
            typ = r.get('type','?')
            score = r.get('score', 0)
            img_tag = f"<img src='file://{r['image_path']}' width=120 />" if r.get('image_path') else ""
        else:
            snippet = str(r)[:400].replace('\n',' ')
            source = page = typ = '?'
            score = 0
            img_tag = ""
        
        ranked_html += f"""
        <div style='border:1px solid #ddd;padding:8px;margin:6px;border-radius:6px'>
            <b>Rank {i}</b> — Type: {typ} — Score: {score:.3f}<br>
            Source: {source} (page {page})<br>
            {img_tag}<br>
            <small>{snippet}</small>
        </div>
        """
    return ranked_html


# ===== ENHANCED GRADIO INTERFACE =====

with gr.Blocks() as demo:
    gr.Markdown("# Enhanced Multimodal Retrieval + RAG with Evaluation")
    
    # ==== Tab 1: Search Interface ====
    with gr.Tab("Search Interface"):
        with gr.Row():
            with gr.Column(scale=3):
                query_input = gr.Textbox(
                    label="Text query",
                    placeholder="Ask about financial figures, policies, etc."
                )
                image_input = gr.Image(type="pil", label="Upload image (optional)")
                
                top_k = gr.Slider(
                    minimum=1, maximum=10, value=5, step=1,
                    label="Top K"
                )
                
                use_llm = gr.Checkbox(
                    label="Use LLM to synthesize answer",
                    value=True
                )
                
                with gr.Accordion("Advanced Options", open=False):
                    prompting_technique = gr.Dropdown(
                        choices=["zero_shot", "few_shot", "cot", "zero_shot_cot"],
                        value="zero_shot",
                        label="Prompting Technique"
                    )
                    compare_techniques = gr.Checkbox(
                        label="Compare all prompting techniques",
                        value=False
                    )
                
                submit = gr.Button("Search")
            
            with gr.Column(scale=2):
                gr.Markdown("**Results (ranked)**")
                results_output = gr.HTML()
                
                gr.Markdown("**Generated Answer**")
                answer_output = gr.Textbox(lines=6)
                
                latency_output = gr.Textbox(
                    lines=1, label="Performance"
                )
                
                comparison_output = gr.HTML(label="Technique Comparison")
    
    # ==== Tab 2: Retrieval Evaluation ====
    with gr.Tab("Retrieval Evaluation"):
        gr.Markdown("## Run Comprehensive Retrieval Evaluation")
        
        eval_button = gr.Button("Run Evaluation")
        
        eval_results = gr.JSON(label="Evaluation Results")
        eval_plot = gr.Image(label="Evaluation Metrics Plot")
    
    # ==== Tab 3: Prompting Analysis ====
    with gr.Tab("Prompting Analysis"):
        gr.Markdown("## Analyze Prompting Techniques")
        
        analyze_button = gr.Button("Run Prompting Analysis")
        analysis_results = gr.JSON(label="Analysis Results")
    
    
    # ===== EVENT HANDLERS =====
    submit.click(
        fn=enhanced_query_handler,
        inputs=[query_input, image_input, top_k, use_llm, prompting_technique, compare_techniques],
        outputs=[results_output, answer_output, latency_output, comparison_output]
    )

    eval_button.click(
        fn=run_retrieval_evaluation,
        outputs=[eval_results]
    )

    analyze_button.click(
        fn=lambda: analyze_prompting_impact(
            ["financial performance", "academic policies", "research methodology"]
        ),
        outputs=[analysis_results]
    )

# IMPORTANT: Enable external Gradio URL in Kaggle
demo.launch(share=True)


# Extra: UMAP visualization helper (call manually if desired)
def plot_umap(save_path=os.path.join(WORK_DIR, "umap.png")):
    emb = np.load(EMBED_NPY)
    reducer = umap.UMAP(n_components=2, random_state=42)
    proj = reducer.fit_transform(emb)
    plt.figure(figsize=(10,8))
    plt.scatter(proj[:,0], proj[:,1], s=6)
    plt.title("Embedding UMAP")
    plt.savefig(save_path)
    print("UMAP saved to", save_path)

print("Notebook ready. Use the Gradio UI above to query text or images.")

2025-11-23 09:39:59.777592: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763890800.018842      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763890800.092592      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading text embedder: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


OpenAI error: 

You tried to access openai.Completion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742



Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


OpenAI error: 

You tried to access openai.Completion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742



Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
